In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║       BrailleVision Hackathon 2026 — Colab Training          ║
# ║   Run each cell one by one in Google Colab (T4 GPU)          ║
# ╚══════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────
# CELL 1 — Check GPU
# ─────────────────────────────────────────────
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


Tue May 26 11:30:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Mount Google Drive
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Extract TAR to Colab's fast local SSD
# (This is the KEY step — Colab SSD is 10x faster than Drive for training)
# ─────────────────────────────────────────────
import os, tarfile, time

# ✏️ Change this to your actual TAR file path in Drive
TAR_PATH = "/content/drive/MyDrive/Colab Notebooks/Braille Hackathon/clean_dataset.tar"

EXTRACT_TO = "/content/dataset"   # Colab local SSD — very fast I/O
os.makedirs(EXTRACT_TO, exist_ok=True)

print("Extracting TAR... (takes ~3–5 mins for 300k images)")
t0 = time.time()
with tarfile.open(TAR_PATH, 'r') as tar:
    tar.extractall(EXTRACT_TO)
print(f"Done in {(time.time()-t0)/60:.1f} minutes")

# Check structure
!ls /content/dataset/
!ls /content/dataset/clean_dataset/   # should show: train/ val/ test/


Extracting TAR... (takes ~3–5 mins for 300k images)


/tmp/ipykernel_1608/2900084313.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_TO)


Done in 1.2 minutes
clean_dataset
dataset_stats.json  phase1_stats.json  raw			  test	 val
labels.csv	    phase2_stats.json  synthetic_sample_grid.png  train


In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Install dependencies
# ─────────────────────────────────────────────
!pip install -q timm wandb  # timm has better EfficientNet, wandb for tracking


In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Imports & Config
# ─────────────────────────────────────────────
import json
import time
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
import timm

# Paths — all on fast local SSD now
DATA_DIR  = Path("/content/dataset/clean_dataset")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"
SAVE_DIR  = Path("/content/drive/MyDrive/BrailleVision/models")  # save back to Drive
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Training config
IMG_SIZE   = 128
BATCH_SIZE = 256    # T4 has 15GB — use 256 for speed
NUM_EPOCHS = 10     # 30 epoch
LR         = 1e-3
DEVICE     = torch.device('cuda')

CLASS_TO_CHAR = {
    'a':'a','b':'b','c':'c','d':'d','e':'e','f':'f','g':'g','h':'h',
    'i':'i','j':'j','k':'k','l':'l','m':'m','n':'n','o':'o','p':'p',
    'q':'q','r':'r','s':'s','t':'t','u':'u','v':'v','w':'w','x':'x',
    'y':'y','z':'z',
    '0':'0','1':'1','2':'2','3':'3','4':'4',
    '5':'5','6':'6','7':'7','8':'8','9':'9',
    'period':'.','comma':',','exclamation':'!','question':'?',
    'colon':':','semicolon':';','hyphen':'-','apostrophe':"'",
    'capital_sign':'[CAP]','number_sign':'[NUM]',
}

print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Transforms & Datasets
# ─────────────────────────────────────────────
train_tfm = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomRotation(10),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.4),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.GaussianBlur(3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_tfm = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfm)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=val_tfm)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=val_tfm)

NUM_CLASSES = len(train_ds.classes)
print(f"Classes : {NUM_CLASSES}")
print(f"Train   : {len(train_ds):,}")
print(f"Val     : {len(val_ds):,}")
print(f"Test    : {len(test_ds):,}")

# Weighted sampler for class imbalance
class_counts  = np.array([train_ds.targets.count(i) for i in range(NUM_CLASSES)])
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_ds.targets]
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), len(train_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

# Save class map to Drive
idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
idx_to_char  = {v: CLASS_TO_CHAR.get(k, k) for k, v in train_ds.class_to_idx.items()}
with open(SAVE_DIR / 'class_map.json', 'w') as f:
    json.dump({'idx_to_class': idx_to_class, 'idx_to_char': idx_to_char}, f)
print("Class map saved to Drive ✅")


Classes : 46
Train   : 280,217
Val     : 35,009
Test    : 34,999
Class map saved to Drive ✅


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Model (EfficientNet-B3 via timm)
# ─────────────────────────────────────────────
def build_model(num_classes):
    model = timm.create_model(
        'efficientnet_b3',
        pretrained=True,
        num_classes=num_classes,
        drop_rate=0.3,
        drop_path_rate=0.2,
    )
    return model

model = build_model(NUM_CLASSES).to(DEVICE)
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model  : EfficientNet-B3 via timm")
print(f"Params : {params:.1f}M")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Model  : EfficientNet-B3 via timm
Params : 10.8M


In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Training Setup
# ─────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = GradScaler()

def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for imgs, labels in loader:[[]]
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (model(imgs).argmax(1) == labels).sum().item() if False else 0
        total    += imgs.size(0)
    return loss_sum / total

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with autocast():
            preds = model(imgs).argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
    return correct / total


/tmp/ipykernel_1608/3103564675.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()


In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — TRAINING LOOP
# ─────────────────────────────────────────────
best_val_acc = 0.0
history = []
patience_count = 0
PATIENCE = 5

print("="*55)
print("TRAINING STARTED")
print("="*55)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
    vl_acc  = eval_epoch(model, val_loader)
    scheduler.step()
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'val_acc': vl_acc})
    tag = ""

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': vl_acc,
            'idx_to_char': idx_to_char,
        }, SAVE_DIR / 'best_model.pth')
        tag = " ✅ SAVED"
        patience_count = 0
    else:
        patience_count += 1

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Loss: {tr_loss:.4f} | "
          f"Val: {vl_acc*100:.2f}% | "
          f"{elapsed:.0f}s{tag}")

    if patience_count >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break

print(f"\nBest Val Accuracy: {best_val_acc*100:.2f}%")


TRAINING STARTED


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_1608/3103564675.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1608/3103564675.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 01/10 | Loss: 0.9273 | Val: 96.47% | 1154s ✅ SAVED
Epoch 02/10 | Loss: 0.8310 | Val: 96.42% | 1080s
Epoch 03/10 | Loss: 0.8229 | Val: 96.56% | 1081s ✅ SAVED
Epoch 04/10 | Loss: 0.8195 | Val: 96.53% | 1082s
Epoch 05/10 | Loss: 0.8180 | Val: 96.70% | 1065s ✅ SAVED
Epoch 06/10 | Loss: 0.8127 | Val: 96.70% | 1069s
Epoch 07/10 | Loss: 0.8098 | Val: 96.78% | 1073s ✅ SAVED
Epoch 08/10 | Loss: 0.8072 | Val: 96.78% | 1073s
Epoch 09/10 | Loss: 0.8066 | Val: 96.81% | 1073s ✅ SAVED
Epoch 10/10 | Loss: 0.8040 | Val: 96.82% | 1087s ✅ SAVED

Best Val Accuracy: 96.82%


In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Test Accuracy
# ─────────────────────────────────────────────
ckpt = torch.load(SAVE_DIR / 'best_model.pth')
model.load_state_dict(ckpt['model_state_dict'])
test_acc = eval_epoch(model, test_loader)
print(f"Test Accuracy: {test_acc*100:.2f}%")


/tmp/ipykernel_1608/3103564675.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Test Accuracy: 96.81%


In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Export TorchScript (for your FastAPI)
# ─────────────────────────────────────────────
model.eval()
traced = torch.jit.trace(model, torch.randn(1,3,128,128).to(DEVICE))
traced.save(SAVE_DIR / 'braille_scripted.pt')
print(f"TorchScript saved to Drive ✅")
print(f"\nDownload these 2 files to your PC:")
print(f"  → MyDrive/BrailleVision/models/best_model.pth")
print(f"  → MyDrive/BrailleVision/models/class_map.json")

TorchScript saved to Drive ✅

Download these 2 files to your PC:
  → MyDrive/BrailleVision/models/best_model.pth
  → MyDrive/BrailleVision/models/class_map.json


# Test the model in Google Colab


In [ ]:
# ─────────────────────────────────────────────
# CELL — Test Model on a Braille Image
# Upload any Braille cell image and see result
# ─────────────────────────────────────────────

import torch
import torch.nn as nn
import json
import timm
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from google.colab import files

# ── Load model & class map ──
SAVE_DIR   = "models"
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(f"{SAVE_DIR}/class_map.json") as f:
    data = json.load(f)
idx_to_char = {int(k): v for k, v in data['idx_to_char'].items()}
NUM_CLASSES = len(idx_to_char)

ckpt  = torch.load(f"{SAVE_DIR}/best_model.pth", map_location=DEVICE)
model = timm.create_model('efficientnet_b3', pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(ckpt['model_state_dict'])
model.eval().to(DEVICE)
print(f"✅ Model loaded | Val Acc when saved: {ckpt['val_acc']*100:.2f}%")

# ── Transform ──
tfm = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── Upload & Predict ──
print("\nUpload a Braille cell image...")
uploaded = files.upload()   # opens file picker

for fname, fbytes in uploaded.items():
    img = Image.open(__import__('io').BytesIO(fbytes)).convert('RGB')

    x      = tfm(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs  = torch.softmax(model(x), dim=1)[0]
    top5_v, top5_i = probs.topk(5)

    pred_char = idx_to_char[top5_i[0].item()]
    pred_conf = top5_v[0].item() * 100

    # ── Display ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.imshow(img, cmap='gray')
    ax1.set_title(f'Input: {fname}', fontsize=10)
    ax1.axis('off')

    labels = [idx_to_char[i.item()] for i in top5_i]
    values = [v.item()*100 for v in top5_v]
    colors = ['#2ecc71' if i==0 else '#3498db' for i in range(5)]
    bars   = ax2.barh(labels[::-1], values[::-1], color=colors[::-1])
    ax2.set_xlabel('Confidence %')
    ax2.set_title('Top 5 Predictions')
    for bar, val in zip(bars, values[::-1]):
        ax2.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)
    ax2.set_xlim(0, 110)

    plt.suptitle(f'Prediction: "{pred_char}"  |  Confidence: {pred_conf:.1f}%',
                 fontsize=14, fontweight='bold',
                 color='green' if pred_conf > 70 else 'orange')
    plt.tight_layout()
    plt.show()

    print(f"\n{'='*40}")
    print(f"  Predicted : {pred_char}")
    print(f"  Confidence: {pred_conf:.1f}%")
    print(f"  Status    : {'✅ Confident' if pred_conf > 70 else '⚠️ Low confidence'}")
    print(f"{'='*40}")